# KAN-15 - Resumos de Variáveis Quantitativas

| | |
|---|---|
| **Projeto** | Análise Estatística — Spaceship Titanic (Pós-Graduação em Deep Learning — CIn/UFPE) |
| **Ticket** | [KAN-15](https://megamatic.atlassian.net/browse/KAN-15) — Produzir resumos de variáveis quantitativas |
| **Prioridade** | High |
| **Responsável** | Helton Gomes Soares de Lima |
| **Revisora** | Luciana Almeida |
| **Dependência** | [KAN-12](https://megamatic.atlassian.net/browse/KAN-12) — Variáveis derivadas (concluída) |
| **Bloqueia** | [KAN-16](https://megamatic.atlassian.net/browse/KAN-16) (gráficos) e [KAN-17](https://megamatic.atlassian.net/browse/KAN-17) (limites de outliers) |
| **Base de dados** | `data/processed/spaceship_titanic_clean.csv` — 8.693 passageiros × 22 colunas, 0 valores ausentes |
| **Critério de aceitação** | Tabela de medidas descritivas quantitativas publicada e validada |

## Objetivo da análise

Caracterizar numericamente o público a bordo em **três dimensões**: **tendência central** (onde os dados se concentram), **dispersão** (o quanto variam) e **assimetria** (se a cauda puxa para um lado). O produto é a tabela oficial de estatísticas descritivas que sustenta as etapas seguintes — as visualizações da KAN-16, os limites de outliers da KAN-17 e o texto consolidado da KAN-18 — além de servir de base para decisões de precificação e segmentação de clientes.

## Nota metodológica: os zeros **não** são ausências

Nas rubricas de consumo (`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`), **zero é um valor observado e legítimo** — passageiros em criossono, crianças ou quem simplesmente não consumiu. Excluir essas linhas não "corrige" a média: ela apenas **troca a população analisada** de *todos os passageiros* para *passageiros que gastaram*. Por isso este notebook apresenta três leituras explícitas, sem remover nenhum registro:

| Leitura | O que responde | Uso no relatório |
|---|---|---|
| **A. População** (n = 8.693, zeros incluídos) | "Como o público se comporta, em média?" | estatística oficial da base |
| **B. Entre gastadores** (`valor > 0`) | "Quanto gasta quem efetivamente gasta?" | dimensionamento de ticket por rubrica |
| **C. Impacto dos zeros** | "Quanto os zeros deslocam a média e a mediana?" | evita a leitura ingênua de média |

> **Regra de ouro:** ao citar qualquer rubrica de gasto, reporte **mediana e média juntas**. Média sem mediana, nesse dataset, produz conclusão errada — e é exatamente o tipo de leitura que o PDF do trabalho penaliza.


In [1]:
# =============================================================================
# KAN-15 — Estatísticas descritivas das variáveis quantitativas
#
# Três leituras complementares (nenhum registro é removido):
#   A) População  : todas as 8.693 observações (zeros incluídos)
#   B) Gastadores : subconjunto com valor > 0, por rubrica
#   C) Impacto    : deslocamento provocado pelos zeros
# =============================================================================
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

# --- 1. Localização resiliente da base tratada -------------------------------
CANDIDATOS = [
    Path("../data/processed/spaceship_titanic_clean.csv"),
    Path("data/processed/spaceship_titanic_clean.csv"),
]
DATA_PATH = next((caminho for caminho in CANDIDATOS if caminho.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Base tratada não encontrada. Execute o notebook a partir da pasta "
        "notebooks/ ou da raiz do projeto; no Colab, faça upload de "
        "data/processed/spaceship_titanic_clean.csv."
    )

df = pd.read_csv(DATA_PATH)

# --- 2. Contratos de integridade (herdados das KAN-11/KAN-12) ----------------
assert df.shape == (8693, 22), f"Shape inesperado: {df.shape}"
assert df.isna().sum().sum() == 0, "A base tratada não deve conter valores ausentes"
assert df["PassengerId"].is_unique, "PassengerId deve ser chave única"

# --- 3. Variáveis do ticket ---------------------------------------------------
# Age + 5 rubricas + TotalSpend (derivada = soma das rubricas) + GroupSize.
QUANTITATIVAS = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "TotalSpend",
    "GroupSize",
]
GASTOS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]


# --- 4. Utilitários de formatação (padrão pt-BR) ------------------------------
def pct(valor: float) -> str:
    """Percentual com vírgula decimal."""
    return f"{valor:.2f}%".replace(".", ",")


def brl(valor: float) -> str:
    """Valor monetário no padrão brasileiro."""
    return f"US$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def num(valor: float, casas: int = 2) -> str:
    """Número decimal no padrão brasileiro (vírgula decimal)."""
    return f"{valor:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def classificar_assimetria(skew: float) -> str:
    """Regra de bolso: |skew| < 0,5 simétrica; 0,5-1 moderada; > 1 forte."""
    a = abs(skew)
    if a < 0.5:
        return "Aproximadamente simétrica"
    if a < 1.0:
        return "Assimetria moderada"
    return "Assimetria forte (cauda à direita)"


# =============================================================================
# LEITURA A — População completa (n = 8.693)
# =============================================================================
linhas = []
for var in QUANTITATIVAS:
    s = df[var].astype(float)
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    media, mediana = s.mean(), s.median()
    dp = s.std(ddof=1)  # desvio-padrão AMOSTRAL

    linhas.append({
        "Variável": var,
        "N": int(s.count()),
        "Média": media,
        "Mediana": mediana,
        "Média - Mediana": media - mediana,
        "Desvio-Padrão (s)": dp,
        "Variância (s²)": s.var(ddof=1),
        "Mínimo": s.min(),
        "Q1 (25%)": q1,
        "Q3 (75%)": q3,
        "Máximo": s.max(),
        "Amplitude Total": s.max() - s.min(),
        "IQR (Q3-Q1)": q3 - q1,
        "Coef. Variação (%)": (dp / media * 100) if media else np.nan,
        "Zeros (n)": int((s == 0).sum()),
        "% Zeros": (s == 0).mean() * 100,
        "Assimetria (skew)": s.skew(),
    })

tabela_geral = pd.DataFrame(linhas)
tabela_geral["Leitura da Assimetria"] = tabela_geral["Assimetria (skew)"].apply(classificar_assimetria)

# Formatação apenas para exibição (os dados originais permanecem intactos)
exibicao_geral = tabela_geral.copy()
colunas_float = [
    "Média", "Mediana", "Média - Mediana", "Desvio-Padrão (s)", "Variância (s²)",
    "Mínimo", "Q1 (25%)", "Q3 (75%)", "Máximo", "Amplitude Total",
    "IQR (Q3-Q1)", "Coef. Variação (%)", "% Zeros", "Assimetria (skew)",
]
exibicao_geral[colunas_float] = exibicao_geral[colunas_float].round(2)

print("=" * 100)
print("LEITURA A — POPULAÇÃO COMPLETA (zeros incluídos)   n =", f"{len(df):,}".replace(",", "."))
print("=" * 100)
display(exibicao_geral.drop(columns=["Leitura da Assimetria"]))
print("\nLeitura da assimetria:")
for var, leitura in zip(tabela_geral["Variável"], tabela_geral["Leitura da Assimetria"]):
    print(f"  {var:<14} {leitura}")


# =============================================================================
# LEITURA B — Entre gastadores (valor > 0)
# =============================================================================
linhas_b = []
for var in GASTOS + ["TotalSpend"]:
    s = df.loc[df[var] > 0, var].astype(float)
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    linhas_b.append({
        "Variável": var,
        "N (> 0)": int(s.size),
        "% da amostra": s.size / len(df) * 100,
        "Média (> 0)": s.mean(),
        "Mediana (> 0)": s.median(),
        "Desvio-Padrão (> 0)": s.std(ddof=1),
        "Q1 (> 0)": q1,
        "Q3 (> 0)": q3,
        "Máximo (> 0)": s.max(),
        "Assimetria (> 0)": s.skew(),
    })

tabela_gastadores = pd.DataFrame(linhas_b)
exibicao_b = tabela_gastadores.round(2)

print("\n" + "=" * 100)
print("LEITURA B — ENTRE GASTADORES (somente valores > 0)")
print("=" * 100)
display(exibicao_b)


# =============================================================================
# LEITURA C — Impacto dos zeros (quanto a média/mediana se deslocam)
# =============================================================================
linhas_c = []
for var in GASTOS + ["TotalSpend"]:
    s = df[var].astype(float)
    s_pos = s[s > 0]
    media_pop, mediana_pop = s.mean(), s.median()
    media_pos, mediana_pos = s_pos.mean(), s_pos.median()
    zeros_pct = (s == 0).mean() * 100

    linhas_c.append({
        "Variável": var,
        "% Zeros": zeros_pct,
        "Média (população)": media_pop,
        "Média (> 0)": media_pos,
        "Δ% Média": (media_pop - media_pos) / media_pos * 100,
        "Mediana (população)": mediana_pop,
        "Mediana (> 0)": mediana_pos,
        "Δ% Mediana": ((mediana_pop - mediana_pos) / mediana_pos * 100) if mediana_pos else np.nan,
    })

tabela_impacto = pd.DataFrame(linhas_c)
tabela_impacto["Leitura"] = pd.cut(
    tabela_impacto["% Zeros"],
    bins=[-0.01, 20, 40, 100],
    labels=["Diluição baixa", "Diluição moderada", "Diluição forte"],
).astype(str)

print("\n" + "=" * 100)
print("LEITURA C — IMPACTO DOS ZEROS SOBRE AS MEDIDAS")
print("=" * 100)
display(tabela_impacto.round(2))
print(
    "Nota: para variáveis não negativas, Δ% Média = −% Zeros (identidade matemática:\n"
    "a média populacional é a média dos gastadores ponderada pela fração que gastou).\n"
    "A coluna Δ% Mediana é a que traz informação nova."
)


# =============================================================================
# CORRELAÇÃO — matriz numérica (Spearman principal; Pearson como apêndice)
# =============================================================================
# Spearman: baseado em postos, robusto à assimetria e aos valores extremos.
# Pearson: sensível à cauda; serve de comparação/diagnóstico.
matriz_spearman = df[QUANTITATIVAS].corr(method="spearman")
matriz_pearson = df[QUANTITATIVAS].corr(method="pearson")

print("\n" + "=" * 100)
print("CORRELAÇÃO DE SPEARMAN (ρ) — todas as variáveis do ticket")
print("=" * 100)
display(matriz_spearman.round(2))

print("\nCORRELAÇÃO DE PEARSON (r) — apêndice/comparação")
display(matriz_pearson.round(2))

# Pares mais fortes (triângulo superior), para orientar a interpretação
pares = []
for i, var_a in enumerate(QUANTITATIVAS):
    for var_b in QUANTITATIVAS[i + 1:]:
        pares.append({
            "Par": f"{var_a} × {var_b}",
            "Spearman (ρ)": matriz_spearman.loc[var_a, var_b],
            "Pearson (r)": matriz_pearson.loc[var_a, var_b],
        })

tabela_pares = pd.DataFrame(pares)
tabela_pares["|ρ|"] = tabela_pares["Spearman (ρ)"].abs()
tabela_pares = tabela_pares.sort_values("|ρ|", ascending=False).drop(columns=["|ρ|"])

print("\nTOP 10 pares por |Spearman|:")
display(tabela_pares.head(10).round(3))


# =============================================================================
# CONFERÊNCIA RÁPIDA — valores-chave para validação com a revisora
# =============================================================================
print("\n" + "=" * 100)
print("CONFERÊNCIA — valores-chave")
print("=" * 100)
print(f"Age            média {num(df['Age'].mean())} | mediana {num(df['Age'].median(), 1)} | skew {num(df['Age'].skew())}")
print(f"TotalSpend     média {brl(df['TotalSpend'].mean())} | mediana {brl(df['TotalSpend'].median())} | skew {num(df['TotalSpend'].skew())}")
print(f"TotalSpend>0   média {brl(df.loc[df['TotalSpend'] > 0, 'TotalSpend'].mean())} | mediana {brl(df.loc[df['TotalSpend'] > 0, 'TotalSpend'].median())}")
print(f"GroupSize      média {num(df['GroupSize'].mean())} | mediana {num(df['GroupSize'].median(), 0)} | máximo {df['GroupSize'].max()}")
print(f"Zeros          " + " | ".join(f"{c} {pct((df[c] == 0).mean() * 100)}" for c in GASTOS))
print(f"\nPassageiros sozinhos (IsAlone): {pct(df['IsAlone'].mean() * 100)}")
print(f"Crianças (Age <= 12): {int((df['Age'] <= 12).sum())} passageiros ({pct((df['Age'] <= 12).mean() * 100)})")


LEITURA A — POPULAÇÃO COMPLETA (zeros incluídos)   n = 8.693


,Variável,N,Média,Mediana,Média - Mediana,Desvio-Padrão (s),Variância (s²),Mínimo,Q1 (25%),Q3 (75%),Máximo,Amplitude Total,IQR (Q3-Q1),Coef. Variação (%),Zeros (n),% Zeros,Assimetria (skew)
0,Age,8693,28.78,27.0,1.78,14.35,206.01,0.0,20.0,37.0,79.0,79.0,17.0,49.87,178,2.05,0.43
1,RoomService,8693,221.58,0.0,221.58,660.80,436653.34,0.0,0.0,43.0,14327.0,14327.0,43.0,298.22,5671,65.24,6.38
2,FoodCourt,8693,451.40,0.0,451.40,1596.28,2548110.72,0.0,0.0,67.0,29813.0,29813.0,67.0,353.63,5549,63.83,7.17
3,ShoppingMall,8693,169.95,0.0,169.95,597.95,357544.52,0.0,0.0,23.0,23492.0,23492.0,23.0,351.84,5720,65.80,12.77
4,Spa,8693,305.45,0.0,305.45,1125.46,1266666.31,0.0,0.0,56.0,22408.0,22408.0,56.0,368.47,5412,62.26,7.72
5,VRDeck,8693,299.09,0.0,299.09,1134.08,1286143.48,0.0,0.0,41.0,24133.0,24133.0,41.0,379.18,5598,64.40,7.90
6,TotalSpend,8693,1447.46,717.0,730.46,2808.28,7886438.03,0.0,0.0,1444.0,35987.0,35987.0,1444.0,194.01,3644,41.92,4.40
7,GroupSize,8693,2.04,1.0,1.04,1.60,2.55,1.0,1.0,3.0,8.0,7.0,2.0,78.42,0,0.00,1.89



Leitura da assimetria:
  Age            Aproximadamente simétrica
  RoomService    Assimetria forte (cauda à direita)
  FoodCourt      Assimetria forte (cauda à direita)
  ShoppingMall   Assimetria forte (cauda à direita)
  Spa            Assimetria forte (cauda à direita)
  VRDeck         Assimetria forte (cauda à direita)
  TotalSpend     Assimetria forte (cauda à direita)
  GroupSize      Assimetria forte (cauda à direita)

LEITURA B — ENTRE GASTADORES (somente valores > 0)


,Variável,N (> 0),% da amostra,Média (> 0),Mediana (> 0),Desvio-Padrão (> 0),Q1 (> 0),Q3 (> 0),Máximo (> 0),Assimetria (> 0)
0,RoomService,3022,34.76,637.39,303.5,995.60,31.0,820.75,14327.0,4.07
1,FoodCourt,3144,36.17,1248.10,369.0,2460.11,33.0,1237.50,29813.0,4.42
2,ShoppingMall,2973,34.20,496.93,180.0,939.76,20.0,684.00,23492.0,8.55
3,Spa,3281,37.74,809.28,210.0,1717.21,22.0,753.00,22408.0,4.86
4,VRDeck,3095,35.60,840.05,243.0,1777.24,24.0,793.00,24133.0,4.84
5,TotalSpend,5049,58.08,2492.13,1192.0,3312.92,798.0,2654.00,35987.0,3.70



LEITURA C — IMPACTO DOS ZEROS SOBRE AS MEDIDAS


...


CORRELAÇÃO DE PEARSON (r) — apêndice/comparação


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,TotalSpend,GroupSize
Age,1.00,0.07,0.13,0.03,0.12,0.10,0.19,-0.18
RoomService,0.07,1.00,-0.02,0.06,0.01,-0.02,0.23,-0.04
FoodCourt,0.13,-0.02,1.00,-0.01,0.22,0.23,0.74,0.03
ShoppingMall,0.03,0.06,-0.01,1.00,0.01,-0.01,0.22,-0.04
Spa,0.12,0.01,0.22,0.01,1.00,0.15,0.59,0.02
VRDeck,0.10,-0.02,0.23,-0.01,0.15,1.00,0.59,0.01
TotalSpend,0.19,0.23,0.74,0.22,0.59,0.59,1.00,0.01
GroupSize,-0.18,-0.04,0.03,-0.04,0.02,0.01,0.01,1.00



TOP 10 pares por |Spearman|:


,Par,Spearman (ρ),Pearson (r)
16,FoodCourt × TotalSpend,0.673,0.743
23,Spa × TotalSpend,0.668,0.593
25,VRDeck × TotalSpend,0.639,0.588
11,RoomService × TotalSpend,0.535,0.234
15,FoodCourt × VRDeck,0.517,0.228
20,ShoppingMall × TotalSpend,0.511,0.222
14,FoodCourt × Spa,0.492,0.222
22,Spa × VRDeck,0.454,0.150
8,RoomService × ShoppingMall,0.453,0.056
5,Age × TotalSpend,0.264,0.187



CONFERÊNCIA — valores-chave
Age            média 28,78 | mediana 27,0 | skew 0,43
TotalSpend     média US$ 1.447,46 | mediana US$ 717,00 | skew 4,40
TotalSpend>0   média US$ 2.492,13 | mediana US$ 1.192,00
GroupSize      média 2,04 | mediana 1 | máximo 8
Zeros          RoomService 65,24% | FoodCourt 63,83% | ShoppingMall 65,80% | Spa 62,26% | VRDeck 64,40%

Passageiros sozinhos (IsAlone): 55,27%
Crianças (Age <= 12): 806 passageiros (9,27%)


## Template de Interpretação para o Relatório

> Preencha os blocos abaixo **depois** de executar a célula de código. Cada número citado deve vir da tabela correspondente (A, B ou C). Este texto alimenta diretamente a seção "Análise Descritiva e Visual" do relatório final (KAN-18).

---

### Análise de Idade

[Inserir interpretação aqui com base nos números gerados]

*Roteiro:* informe média, mediana e DP; compare média e mediana para concluir sobre a assimetria; comente o coeficiente de variação; e **não esqueça das crianças** (há 178 registros com idade 0 e cerca de 9% da base com até 12 anos). Se a média e a mediana estiverem próximas, diga explicitamente que a média é uma síntese adequada.

### Análise de Gastos

#### (a) Rubricas individuais — leitura populacional

[Inserir interpretação aqui com base nos números gerados]

*Roteiro:* para cada rubrica, cite mediana e média **juntas**; destaque o percentual de zeros; explique que mediana zero não significa "gastou pouco", mas "a maioria não gastou".

#### (b) Rubricas individuais — entre gastadores

[Inserir interpretação aqui com base nos números gerados]

*Roteiro:* aqui está o ticket médio real por serviço. Compare com a leitura (a) e quantifique a diferença. Deixe claro que esta é a média **de quem consome**, não do passageiro típico.

#### (c) Zero-inflação e suas consequências

[Inserir interpretação aqui com base nos números gerados]

*Roteiro:* use a tabela C para mostrar quanto os zeros deslocam a média (Δ%). Conclua por que qualquer decisão de precificação deve partir da mediana ou da leitura entre gastadores. **Atenção:** para variáveis não negativas, `Δ% Média = −% Zeros` é uma identidade matemática (a média populacional é a média dos gastadores ponderada pela fração que gastou) — não venda isso como descoberta empírica. O número informativo dessa tabela é o **Δ% da Mediana**.

### Análise de TotalSpend e GroupSize

[Inserir interpretação aqui com base nos números gerados]

*Roteiro:* `TotalSpend` concentra a conclusão do capítulo — compare média (população) e média entre gastadores, e a razão média/mediana. Em `GroupSize`, lembre que é variável discreta (1 a 8): comente a concentração em viajantes sozinhos e o efeito dos grupos grandes na média.

### Correlação de Gastos

[Inserir interpretação aqui com base nos números gerados]

*Roteiro:* identifique os pares mais fortes na matriz de Spearman; explique o **co-consumo** (quem usa um serviço tende a usar outros); comente a correlação de cada rubrica com `TotalSpend` como esperada por construção; verifique a correlação com `Age` (tende a ser fraca). Se Pearson for muito maior que Spearman em algum par, isso indica que poucos valores extremos estão comandando a relação — reporte Spearman e explique.

---

## Dicas para escrever um bom relatório

| O que observar | Por que importa |
|---|---|
| Assimetria de `Age` próxima de zero | indica que média e mediana são equivalentes; a média pode ser citada isoladamente |
| Assimetria > 1 nas rubricas | obriga o par mediana + média; a média sozinha superestima o cliente típico |
| `% Zeros` acima de 60% | explica por que a mediana é zero e por que existe a leitura "entre gastadores" |
| Razão média/mediana em `TotalSpend` | mede a força da cauda; se ≈ 2, a média não representa o passageiro mediano |
| `Δ% Média` igual a `−% Zeros` | é identidade matemática, não achado empírico; o dado novo é o `Δ% Mediana` |
| Coeficiente de variação alto (> 100%) | sinaliza heterogeneidade; sustenta a recomendação de segmentar |
| Correlações altas entre rubricas | evidência de co-consumo; cuidado para não interpretar como causalidade |
| Correlação de `TotalSpend` com as rubricas | é esperada (soma); não trate como descoberta |
| Diferença Pearson vs. Spearman | diagnóstico de influência de outliers na relação |

### Erros a evitar

- Citar "gasto médio de US$ 221 em RoomService" sem dizer que 65% dos passageiros gastaram zero.
- Chamar os zeros de "dados faltantes" ou removê-los da base.
- Apresentar Pearson em variáveis com cauda longa sem a versão de Spearman.
- Usar "prova", "causa" ou "devido a" — a análise é descritiva, não causal.
- Arredondar para inteiro valores em US$ que perdem a diferença entre média e mediana.

### Checklist do ticket (KAN-15)

- [ ] Tabela A publicada com as 8 variáveis e todas as métricas exigidas (média, mediana, DP, variância, mínimo, Q1, Q3, máximo, amplitude, IQR)
- [ ] Comparação média × mediana com leitura de assimetria por variável
- [ ] Leituras B e C publicadas e rotuladas (zeros **não** removidos)
- [ ] Matriz de correlação (Spearman principal, Pearson em apêndice)
- [ ] Seção de interpretação preenchida com números reais
- [ ] Revisão da Luciana registrada no ticket
